# Volume 5 Drills — Advanced Policy Optimization

Work through each drill problem. Code cells are provided for problems 3, 8, 9, and 13.

## Drill 1 — PPO: What Does "Proximal" Mean?

**Question:** PPO is called **Proximal** Policy Optimization. What does "proximal" refer to?

**Answer (fill in):** ___

<details><summary>Answer</summary>

"Proximal" means keeping the **new policy close to the old policy** during each update. Without this constraint, large gradient steps could move the policy into a bad region from which recovery is slow.

PPO enforces this via **clipping** the probability ratio r(θ) = π_new(a|s) / π_old(a|s) to stay within [1−ε, 1+ε]. This is a simpler, more efficient approximation to TRPO's KL divergence constraint.
</details>

## Drill 2 — PPO Clipping: Compute the Clipped Objective

Given:
- Probability ratio r = 1.5
- Clip ε = 0.2
- Advantage Â = 1.0

The PPO clipped objective for one transition is:

$$L^{CLIP} = \min\left(r \cdot \hat{A},\ \text{clip}(r, 1-\varepsilon, 1+\varepsilon) \cdot \hat{A}\right)$$

Compute the value.

<details><summary>Answer</summary>

- r · Â = 1.5 × 1.0 = **1.5**
- clip(1.5, 0.8, 1.2) = **1.2**
- 1.2 × 1.0 = **1.2**
- min(1.5, 1.2) = **1.2** ← the clipped value is used

The clip kicks in because the ratio exceeded 1+ε=1.2, preventing too large a gradient step.
</details>

## Drill 3 — Code: PPO Clipped Objective for One Transition

In [ ]:
import numpy as np

def ppo_clip_objective(ratio, advantage, clip_eps=0.2):
    """
    Compute PPO clipped surrogate objective for a single transition.
    ratio = π_new(a|s) / π_old(a|s)
    """
    unclipped = ratio * advantage
    clipped = np.clip(ratio, 1.0 - clip_eps, 1.0 + clip_eps) * advantage
    return np.minimum(unclipped, clipped)

# Test cases
test_cases = [
    (1.5, 1.0, 0.2),   # ratio too large, positive advantage
    (0.5, 1.0, 0.2),   # ratio too small, positive advantage
    (1.5, -1.0, 0.2),  # ratio too large, negative advantage
    (0.5, -1.0, 0.2),  # ratio too small, negative advantage
    (1.1, 1.0, 0.2),   # ratio within bounds
]

print(f"{'ratio':>6} {'adv':>6} {'eps':>5} | {'unclipped':>10} {'clipped_r':>10} {'L_CLIP':>8}")
print("-" * 55)
for r, adv, eps in test_cases:
    L = ppo_clip_objective(r, adv, eps)
    clipped_r = np.clip(r, 1-eps, 1+eps)
    print(f"{r:>6.2f} {adv:>6.1f} {eps:>5.1f} | {r*adv:>10.3f} {clipped_r:>10.3f} {L:>8.3f}")

## Drill 4 — GAE: What Does λ Control?

**Question:** In Generalized Advantage Estimation (GAE), what does the parameter **λ** control?

**Answer (fill in):** ___

<details><summary>Answer</summary>

λ controls the **bias-variance tradeoff** in the advantage estimate:

$$\hat{A}_t^{GAE(\gamma, \lambda)} = \sum_{l=0}^{\infty} (\gamma\lambda)^l \delta_{t+l}$$

where δ_t = r_t + γV(s_{t+1}) - V(s_t) is the 1-step TD error.

- **λ = 0:** Pure 1-step TD advantage (low variance, high bias — depends entirely on critic V)
- **λ = 1:** Full Monte Carlo advantage (unbiased, high variance — sums all future rewards)
- **λ ∈ (0,1):** Intermediate tradeoff; **λ = 0.95** is a common default.
</details>

## Drill 5 — SAC: Maximum Entropy Objective

**Question:** What does "maximum entropy" add to the standard RL objective in SAC?

**Answer (fill in):** ___

<details><summary>Answer</summary>

SAC augments the standard expected return objective with an **entropy bonus**:

$$J(\pi) = \sum_t \mathbb{E}\left[ r(s_t, a_t) + \alpha \cdot \mathcal{H}(\pi(\cdot | s_t)) \right]$$

where H(π(·|s)) = -E[log π(a|s)] is the entropy of the policy and α is the temperature.

The entropy bonus encourages the policy to:
1. **Explore** — prefer more uniform (random) actions when uncertain
2. **Capture multimodal behavior** — don't prematurely commit to one action
3. **Robustness** — continue exploring even after finding a good policy
</details>

## Drill 6 — SAC vs PPO: Sample Efficiency

**Question:** Why is SAC typically more **sample efficient** than PPO?

**Answer (fill in):** ___

<details><summary>Answer</summary>

SAC is **off-policy** while PPO is **on-policy**:

| | SAC | PPO |
|---|---|---|
| Policy type | Off-policy | On-policy |
| Data reuse | Yes (replay buffer, each sample used many times) | No (data discarded after each update) |
| Sample efficiency | High | Lower |
| Wall-clock speed | Can be slower per step | Faster per step (parallel envs) |

Because SAC stores transitions in a **replay buffer** and reuses them for many gradient updates, it typically needs far fewer environment interactions to reach a given performance level.
</details>

## Drill 7 — TRPO: What Constraint Does It Enforce?

**Question:** What constraint does TRPO (Trust Region Policy Optimization) enforce during policy updates?

**Answer (fill in):** ___

<details><summary>Answer</summary>

TRPO enforces a **KL divergence constraint** between the old and new policy:

$$\mathbb{E}_s\left[ D_{KL}(\pi_{\text{old}}(\cdot|s) \| \pi_{\text{new}}(\cdot|s)) \right] \leq \delta$$

This trust region ensures policy updates are small enough to maintain the monotonic improvement guarantee. TRPO solves this via conjugate gradient + line search, which is computationally expensive. PPO approximates this constraint more efficiently via clipping.
</details>

## Drill 8 — Code: Entropy of a Discrete Distribution

In [ ]:
import numpy as np

def entropy(probs):
    """Shannon entropy H(p) = -Σ p_i log p_i"""
    probs = np.array(probs, dtype=float)
    # Avoid log(0) by masking zero probabilities
    mask = probs > 0
    return -np.sum(probs[mask] * np.log(probs[mask]))

# Drill problem: distribution [0.5, 0.3, 0.2]
p = [0.5, 0.3, 0.2]
H = entropy(p)
print(f"Entropy of {p} = {H:.6f} nats")
print(f"Max entropy (uniform) = {np.log(len(p)):.6f} nats")
print()

# Compare different distributions
distributions = {
    "Uniform [1/3, 1/3, 1/3]": [1/3, 1/3, 1/3],
    "Skewed [0.5, 0.3, 0.2]": [0.5, 0.3, 0.2],
    "Peaked [0.9, 0.05, 0.05]": [0.9, 0.05, 0.05],
    "Deterministic [1, 0, 0]": [1.0, 0.0, 0.0],
}
for name, dist in distributions.items():
    print(f"{name}: H = {entropy(dist):.4f}")

## Drill 9 — Compute: GAE Advantage for a 3-Step Trajectory

In [ ]:
import numpy as np

def compute_gae(rewards, values, gamma=0.99, lam=0.95):
    """
    Generalized Advantage Estimation.
    rewards: list of rewards r_0, ..., r_{T-1}
    values: list of V(s_0), ..., V(s_T)  -- length T+1, last is V(s_T)
    """
    T = len(rewards)
    advantages = np.zeros(T)
    gae = 0.0
    for t in reversed(range(T)):
        delta = rewards[t] + gamma * values[t + 1] - values[t]
        gae = delta + gamma * lam * gae
        advantages[t] = gae
    return advantages

# Drill problem values
rewards = [0, 0, 1]            # r_0, r_1, r_2
values  = [0.3, 0.4, 0.5, 0.0] # V(s_0), V(s_1), V(s_2), V(s_3=terminal)
gamma   = 0.99
lam     = 0.95

advantages = compute_gae(rewards, values, gamma, lam)

print(f"GAE Advantages (γ={gamma}, λ={lam}):")
for t in range(3):
    delta_t = rewards[t] + gamma * values[t+1] - values[t]
    print(f"  t={t}: δ={delta_t:.4f}  A_GAE={advantages[t]:.6f}")

# Also show 1-step (lambda=0) vs MC (lambda=1) for comparison
print()
for lam_val in [0.0, 0.5, 0.95, 1.0]:
    A = compute_gae(rewards, values, gamma, lam_val)
    print(f"  λ={lam_val}: A_0={A[0]:.4f}")

## Drill 10 — Debug: PPO Ratio Computed from Updated Policy

**Find the bug:**

```python
# Collected rollout with old policy
old_log_probs = collect_rollout(env, policy)  # saved log probs

# PPO update loop (multiple epochs over same data)
for epoch in range(n_epochs):
    for batch in dataloader:
        new_log_probs = policy.log_prob(batch.states, batch.actions)
        # BUG: using current policy for old_log_probs too!
        old_log_probs_batch = policy.log_prob(batch.states, batch.actions)
        ratio = torch.exp(new_log_probs - old_log_probs_batch)
        loss = -ppo_clip_objective(ratio, batch.advantages)
```

**What is the bug?**

<details><summary>Answer</summary>

**Bug:** `old_log_probs_batch` is computed using the **current policy** (being updated), not the policy that collected the data. This means `ratio = exp(log_new - log_new) = 1.0` always — the clipping never activates, and the algorithm degenerates.

**Fix:** Store the old log probabilities **during rollout collection** and pass them through the dataloader:

```python
# During rollout:
with torch.no_grad():
    old_log_probs = policy.log_prob(states, actions)  # frozen!

# During PPO update:
ratio = torch.exp(new_log_probs - batch.old_log_probs)  # use stored values
```
</details>

## Drill 11 — Hyperparameter: PPO clip=0.0

**Question:** PPO's standard clip parameter is ε=0.2. What happens if you set ε=0.0?

**Answer (fill in):** ___

<details><summary>Answer</summary>

With ε=0.0, the clip range becomes [1.0, 1.0]. The ratio is always clipped to exactly 1.0:

$$L^{CLIP} = \min(r \cdot \hat{A},\ 1.0 \cdot \hat{A}) = \hat{A}$$

The gradient of this w.r.t. θ is zero — the policy **cannot be updated at all**. The policy is effectively **frozen**.

Larger ε allows larger policy updates but risks instability. The ε=0.2 default is empirically robust across many tasks.
</details>

## Drill 12 — On-Policy vs Off-Policy

**Question:** What is the fundamental difference between **on-policy** and **off-policy** learning?

**Answer (fill in):** ___

<details><summary>Answer</summary>

| | On-Policy | Off-Policy |
|---|---|---|
| Data source | Must come from the **current policy** | Can come from **any policy** (including past/different ones) |
| Data reuse | No — stale data becomes invalid immediately | Yes — replay buffer stores old transitions |
| Examples | REINFORCE, A2C, PPO | Q-learning, DQN, DDPG, SAC, TD3 |
| Sample efficiency | Lower (can't reuse) | Higher (can reuse) |
| Stability | More stable (no distribution mismatch) | Harder (deadly triad risk) |

On-policy methods must collect new data after each update; off-policy methods can learn from a buffer of past experience.
</details>

## Drill 13 — Code: Clamp/Clip Operation for PPO Ratio

In [ ]:
import numpy as np

def ppo_ratio_clamp(new_log_probs, old_log_probs, clip_eps=0.2):
    """
    Compute probability ratio and clamp to [1-eps, 1+eps].
    Works in log space for numerical stability.
    """
    ratio = np.exp(new_log_probs - old_log_probs)
    clipped_ratio = np.clip(ratio, 1.0 - clip_eps, 1.0 + clip_eps)
    return ratio, clipped_ratio

# Simulate a batch of log probs
np.random.seed(42)
old_lp = np.random.uniform(-2, -0.5, size=8)
new_lp = old_lp + np.random.uniform(-0.5, 0.5, size=8)  # policy has changed

ratios, clipped = ppo_ratio_clamp(new_lp, old_lp)

print(f"{'idx':>4} {'old_lp':>8} {'new_lp':>8} {'ratio':>8} {'clipped':>8} {'was_clipped':>12}")
print("-" * 60)
for i, (olp, nlp, r, cr) in enumerate(zip(old_lp, new_lp, ratios, clipped)):
    clipped_flag = "YES" if abs(r - cr) > 1e-8 else "no"
    print(f"{i:>4} {olp:>8.3f} {nlp:>8.3f} {r:>8.3f} {cr:>8.3f} {clipped_flag:>12}")

## Drill 14 — SAC Temperature: Effect of High α

**Question:** In SAC, the temperature parameter α weights the entropy bonus. What does a **high** α value cause?

**Answer (fill in):** ___

<details><summary>Answer</summary>

A **high** temperature α causes:

1. **More exploration** — the entropy bonus dominates, pushing the policy toward a more **uniform/random** distribution over actions.
2. **Less exploitation** — the policy pays less attention to Q-values relative to entropy.
3. **Slow convergence** — may never commit to a specific action even when the optimal action is clear.

Conversely, α → 0 recovers the standard (non-entropy) RL objective, and the policy becomes deterministic.

SAC often uses **automatic temperature tuning** to target a desired entropy level: `α` is optimized to satisfy H(π(·|s)) ≥ H_target.
</details>

## Drill 15 — Challenge: PPO vs SAC Sample Complexity

**Question:** Compare the theoretical sample complexity of PPO vs SAC. Under what conditions would you prefer each?

**Answer (fill in):** ___

<details><summary>Answer</summary>

**Sample complexity comparison:**

| Criterion | PPO | SAC |
|---|---|---|
| Environment steps to solve | More (on-policy, no reuse) | Fewer (replay buffer) |
| Gradient updates per env step | ~1 epoch | Many (off-policy updates) |
| Wall-clock time | Fast with parallel envs | Slower per env step (more compute) |
| Stability | Very stable | Can diverge with bad hyperparams |

**Prefer PPO when:**
- Environment is cheap to simulate (video games, physics sims)
- Parallelism is available (vectorized envs)
- Discrete or mixed action spaces
- Simplicity and robustness are prioritized

**Prefer SAC when:**
- Environment is expensive (real robot, slow sim)
- Continuous action spaces
- Sample efficiency is the bottleneck
- Exploration is important (entropy bonus helps)

In practice: PPO dominates robotics benchmarks with parallel envs; SAC dominates offline/real-world settings.
</details>